# Ленивый Arsenal

Ноутбук использует две готовые конфигурации: многомодельную `test_playbook_arsenal_router_mode.toml` для Router Mode и `test_playbook_arsenal_model_mode.toml` с одной моделью на сервер для Model Mode. Конструктор только читает TOML и создаёт объектное дерево; ресурсы активируются при первом обращении к модели. Закомментированный метод `arsenal.download()` позволяет при желании заранее скачать все ресурсы без запуска процессов.

In [1]:
# Automatically reload imported modules when their source code changes
%load_ext autoreload
%autoreload 2

# Set the working directory to the ZEMI component root
from pathlib import Path

while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

c:\Users\Axoman\Documents\ZEMI\zemilib_tests\tests
c:\Users\Axoman\Documents\ZEMI\zemilib_tests


WindowsPath('c:/Users/Axoman/Documents/ZEMI/zemilib_tests')

In [2]:
import zemi
from zemi.arsenal import ArsenalSession

## Построение объектного дерева

Следующие ячейки только читают TOML и создают объекты. Они не скачивают модели, не скачивают llama.cpp и не запускают процессы.

In [3]:
router_mode_arsenal = ArsenalSession(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_router_mode.toml"
)
# router_mode_arsenal.download()  # Предварительно скачать все ресурсы.

In [4]:
model_mode_arsenal = ArsenalSession(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_model_mode.toml"
)
# model_mode_arsenal.download()  # Предварительно скачать все ресурсы.

# Begin и end playbook

`playbook.begin` включает ленивый режим и сам ничего не скачивает и не запускает. При `stop_before_begin=True` он только останавливает прежние процессы. Скачивание и запуск происходят при первом обращении к конкретной модели.

### Model Mode: чистый запуск

Рекомендуемый вариант: сначала остановить возможные старые процессы Arsenal, затем обращаться только к нужным моделям и остановить запущенные серверы после playbook.

In [5]:
zemi.arsenal.begin(model_mode_arsenal, 
    stop_before_begin=True,
    llama_router_mode=False,
)

primary_model = model_mode_arsenal.llamas.primary.models.qwen
secondary_model = model_mode_arsenal.llamas.secondary.models.phi

# На этом месте обе модели скачаны и оба сервера готовы.

zemi.arsenal.end(model_mode_arsenal, stop_after_end=True)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ОСТАНОВКА ARSENAL
Llama-серверов в конфигурации: 2
══════════════════════════════════════════════════════════════════════════════
[1/2] primary · 127.0.0.1:8080
    · не запущен
[2/2] secondary · 127.0.0.1:8081
    · не запущен
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal остановлен
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ARSENAL ГОТОВ · MODEL MODE
Llama-серверов в конфигурации: 2
══════════════════════════════════════════════════════════════════════════════
Скачивание и запуск отложены до первого обращения к модели.
Пример: arsenal.llamas["primary"].models["qwen"]
══════════════════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════════════

### Model Mode: запуск без предварительной остановки

Используйте только когда известно, что настроенные порты свободны. Завершение с `False` намеренно оставляет серверы работающими для следующего playbook; последняя строка показывает явную последующую очистку.

In [6]:
zemi.arsenal.begin(model_mode_arsenal, 
    stop_before_begin=False,
    llama_router_mode=False,
)

model_mode_arsenal.llamas.primary.models.qwen

# Серверы остаются доступны после завершения playbook.
zemi.arsenal.end(model_mode_arsenal, stop_after_end=False)

# Выполните позже, когда серверы больше не нужны.
zemi.arsenal.end(model_mode_arsenal, stop_after_end=True)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ARSENAL ГОТОВ · MODEL MODE
Llama-серверов в конфигурации: 2
══════════════════════════════════════════════════════════════════════════════
Скачивание и запуск отложены до первого обращения к модели.
Пример: arsenal.llamas["primary"].models["qwen"]
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ЛЕНИВАЯ АКТИВАЦИЯ МОДЕЛИ
Модель: primary/qwen · qwen3.5-4b
Llama:  llama:b9222 · 127.0.0.1:8080
TOML:   @comp/tests/playbook_arsenal/test_playbook_arsenal_model_mode.toml
══════════════════════════════════════════════════════════════════════════════
[1/3] llama.cpp llama:b9222 уже подготовлен
[2/3] Модель primary/qwen уже подготовлена
[3/3] Запускаю primary с моделью qwen...
    ✓ сервер готов · PID 9624
═════════════════════════════════════════════════════════════════════════

## Router Mode: чистый запуск

При первом обращении запускается родительский router с INI-пресетом, в котором заранее указаны пути всех моделей сервера. Последующие модели скачиваются и загружаются через `/models/load` без перезапуска router.

In [7]:
zemi.arsenal.begin(router_mode_arsenal, 
    stop_before_begin=True,
    llama_router_mode=True,
)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ОСТАНОВКА ARSENAL
Llama-серверов в конфигурации: 2
══════════════════════════════════════════════════════════════════════════════
[1/2] primary · 127.0.0.1:8080
    · не запущен
[2/2] secondary · 127.0.0.1:8081
    · не запущен
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal остановлен
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ARSENAL ГОТОВ · ROUTER MODE
Llama-серверов в конфигурации: 2
══════════════════════════════════════════════════════════════════════════════
Скачивание и запуск отложены до первого обращения к модели.
Пример: arsenal.llamas["primary"].models["qwen"]
══════════════════════════════════════════════════════════════════════════════


### Объектная модель запущенного Arsenal

После `playbook.begin` включена ленивая активация, но серверы ещё не запущены. Первое обращение по цепочке `llamas → models` готовит выбранную модель. На каждом уровне объект можно получить по индексу, строковому имени или через точку; свойство `config` содержит исходную TOML-таблицу.

In [8]:
# Llama-сервер: индекс, имя и точечная запись.
primary_by_index = router_mode_arsenal.llamas[0]
primary_by_name = router_mode_arsenal.llamas["primary"]
primary_by_dot = router_mode_arsenal.llamas.primary
assert primary_by_index is primary_by_name is primary_by_dot

# Модель: те же три способа доступа.
qwen_by_index = primary_by_dot.models[0]
qwen_by_name = primary_by_dot.models["qwen"]
qwen_by_dot = primary_by_dot.models.qwen
assert qwen_by_index is qwen_by_name is qwen_by_dot

# Ассистент: те же три способа доступа.
assistant_by_index = qwen_by_dot.assistants[0]
assistant_by_name = qwen_by_dot.assistants["assistant"]
assistant_by_dot = qwen_by_dot.assistants.assistant
assert assistant_by_index is assistant_by_name is assistant_by_dot

# Коллекции сохраняют порядок и поддерживают отрицательные индексы.
assert router_mode_arsenal.llamas[-1].name == "secondary"
assert primary_by_dot.models[-1].name == "smollm"

{
    "llama_names": list(router_mode_arsenal.llamas.keys()),
    "model_names": list(primary_by_dot.models.keys()),
    "assistant_names": list(qwen_by_dot.assistants.keys()),
    "llama_config": primary_by_dot.config,
    "model_config": qwen_by_dot.config,
    "assistant_config": assistant_by_dot.config,
}


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ЛЕНИВАЯ АКТИВАЦИЯ МОДЕЛИ
Модель: primary/qwen · qwen3.5-4b
Llama:  llama:b9222 · 127.0.0.1:8080
TOML:   @comp/tests/playbook_arsenal/test_playbook_arsenal_router_mode.toml
══════════════════════════════════════════════════════════════════════════════
[1/4] Проверяю llama.cpp llama:b9222...
llama.cpp b9222 уже скачан: @inst/_llamas/llama--b9222
[2/4] Проверяю модель primary/qwen...
Модель уже скачана: @inst/_models/hf--bartowski--Qwen_Qwen3.5-4B-GGUF--Qwen_Qwen3.5-4B-Q4_K_M/Qwen_Qwen3.5-4B-Q4_K_M.gguf
[3/4] Запускаю primary в Router Mode...
    Пресет primary: 2 моделей · C:\Users\Axoman\Documents\ZEMI\_tmp\zemi-arsenal-primary.ini
    ✓ сервер готов · PID 18080
[4/4] Загружаю модель qwen3.5-4b в Router Mode...
    ✓ модель qwen3.5-4b загружена
══════════════════════════════════════════════════════════════════════════════
✓ Модель готова: primary/qwen
  Сервер: http://127.0.0.1:8080
═════════

{'llama_names': ['primary', 'secondary'],
 'model_names': ['qwen', 'smollm'],
 'assistant_names': ['assistant', 'json_converter'],
 'llama_config': {'name': 'primary',
  'llama_build': 'llama:b9222',
  'host': '127.0.0.1',
  'port': 8080,
  'startup_timeout': 120.0,
  'models': [{'name': 'qwen',
    'source': 'hf',
    'owner': 'bartowski',
    'repository': 'Qwen_Qwen3.5-4B-GGUF',
    'filename': 'Qwen_Qwen3.5-4B-Q4_K_M.gguf',
    'alias': 'qwen3.5-4b',
    'ctx_size': 8192,
    'threads': 8,
    'threads_batch': 8,
    'reasoning': 'off',
    'assistants': [{'name': 'assistant',
      'prefix': '@comp/tests/zemi_toml/prefixes/qwen-system.md'},
     {'name': 'json_converter',
      'prefix': '@comp/tests/zemi_toml/prefixes/qwen-json.md'}]},
   {'name': 'smollm',
    'source': 'hf',
    'owner': 'bartowski',
    'repository': 'SmolLM2-1.7B-Instruct-GGUF',
    'filename': 'SmolLM2-1.7B-Instruct-Q4_K_M.gguf',
    'alias': 'smollm2-1.7b',
    'ctx_size': 4096,
    'threads': 6,
    'threa

### Завершение playbook

После демонстрации объектной модели останавливаем все серверы из конфигурации.

In [9]:
zemi.arsenal.end(router_mode_arsenal, stop_after_end=True)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ОСТАНОВКА ARSENAL
Llama-серверов в конфигурации: 2
══════════════════════════════════════════════════════════════════════════════
[1/2] primary · 127.0.0.1:8080
    ✓ остановлен · PID 18080
[2/2] secondary · 127.0.0.1:8081
    · не запущен
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal остановлен
══════════════════════════════════════════════════════════════════════════════


## Router Mode: запуск на заведомо свободных портах

Вариант без предварительной остановки полезен, когда состояние окружения контролируется снаружи. Серверы останавливаются после playbook.

In [10]:
zemi.arsenal.begin(router_mode_arsenal, 
    stop_before_begin=False,
    llama_router_mode=True,
)

router_mode_arsenal.llamas.primary.models.qwen
router_mode_arsenal.llamas.primary.models.smollm
# Вторая модель добавлена без перезапуска родительского router.

zemi.arsenal.end(router_mode_arsenal, stop_after_end=True)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ARSENAL ГОТОВ · ROUTER MODE
Llama-серверов в конфигурации: 2
══════════════════════════════════════════════════════════════════════════════
Скачивание и запуск отложены до первого обращения к модели.
Пример: arsenal.llamas["primary"].models["qwen"]
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ЛЕНИВАЯ АКТИВАЦИЯ МОДЕЛИ
Модель: primary/qwen · qwen3.5-4b
Llama:  llama:b9222 · 127.0.0.1:8080
TOML:   @comp/tests/playbook_arsenal/test_playbook_arsenal_router_mode.toml
══════════════════════════════════════════════════════════════════════════════
[1/4] llama.cpp llama:b9222 уже подготовлен
[2/4] Модель primary/qwen уже подготовлена
[3/4] Запускаю primary в Router Mode...
    Пресет primary: 2 моделей · C:\Users\Axoman\Documents\ZEMI\_tmp\zemi-arsenal-primary.ini
    ✓ сер

## Проверка ограничения Model Mode

Исходный Arsenal содержит несколько моделей на сервер. Поэтому попытка запустить его без Router Mode ожидаемо завершается `ValueError` до запуска первого сервера.

In [12]:
try:
    zemi.arsenal.begin(router_mode_arsenal, 
        stop_before_begin=False,
        llama_router_mode=False,
    )
except ValueError as error:
    print(f"Ожидаемая ошибка конфигурации: {error}")

Ожидаемая ошибка конфигурации: Без Router Mode каждый llama-сервер должен содержать ровно одну модель. Нарушение: primary (2 моделей), secondary (2 моделей)
